# Epi Info AI MEANS validation lab — V0.10

Validate the candidate `epi.means` Rust/WebAssembly kernel against the canonical foodborne Age data and independent Python formulas. Passing is evidence, not statistical approval; G5 remains consolidated.

In [ ]:
import csv, hashlib, io, math, statistics
from collections import Counter
from pyodide.http import pyfetch
from js import WebAssembly, Uint8Array
fixture_response = await pyfetch('/validation-fixtures/foodborne-means-v0.10.json')
fixture_response.raise_for_status(); fixture = await fixture_response.json()
data_response = await pyfetch('/examples/foodborne-outbreak-investigation.csv')
data_response.raise_for_status(); data_bytes = await data_response.bytes()
assert hashlib.sha256(data_bytes).hexdigest() == fixture['dataset']['sha256']
records = list(csv.DictReader(io.StringIO(data_bytes.decode('utf-8-sig'))))
ages = [float(record[fixture['request']['sourceHeader']]) for record in records]
wasm_response = await pyfetch('/epi2x2.wasm')
instance = await WebAssembly.instantiate(Uint8Array.new(await wasm_response.buffer()), {})
rust = instance.instance.exports


In [ ]:
def legacy_rank(values, fraction):
    ordered = sorted(values); position = len(ordered) * fraction
    if position.is_integer():
        upper = int(position); return (ordered[upper - 1] + ordered[upper]) / 2
    return ordered[math.ceil(position) - 1]
counts = Counter(ages); mode = min(value for value, count in counts.items() if count == max(counts.values()))
python_results = {'observations': len(ages), 'total': sum(ages), 'mean': statistics.mean(ages), 'variance': statistics.variance(ages), 'standardDeviation': statistics.stdev(ages), 'minimum': min(ages), 'quartile25': legacy_rank(ages, .25), 'median': legacy_rank(ages, .5), 'quartile75': legacy_rank(ages, .75), 'maximum': max(ages), 'mode': mode}
for name, expected in fixture['expected']['statistics'].items(): assert math.isclose(python_results[name], expected, abs_tol=1e-12, rel_tol=0)
print('PASS: independent Python formulas match the foodborne V0.10 anchors')


In [ ]:
rust.means_reset()
for index, value in enumerate(ages): assert rust.means_set_value(index, value) == 1
n = len(ages); assert rust.means_prepare(n) == 1
rust_results = {'observations': n, 'total': float(rust.means_sum(n)), 'mean': float(rust.means_mean(n)), 'variance': float(rust.means_sample_variance(n)), 'standardDeviation': float(rust.means_sample_std_dev(n)), 'minimum': float(rust.means_minimum(n)), 'quartile25': float(rust.means_quartile_25(n)), 'median': float(rust.means_median(n)), 'quartile75': float(rust.means_quartile_75(n)), 'maximum': float(rust.means_maximum(n)), 'mode': float(rust.means_mode(n))}
for name, expected in fixture['expected']['statistics'].items(): assert math.isclose(rust_results[name], expected, abs_tol=1e-12, rel_tol=0)
print('PASS: deployed Rust/WASM matches the foodborne V0.10 anchors')
